# Multi-Agent Ticket Triage System

This notebook preserves the multi-agent learning outcome with current Microsoft Foundry APIs. It creates three specialist prompt-agent versions and one orchestrator prompt agent. The orchestrator exposes three client-side `FunctionTool` definitions; the notebook invokes the requested specialist and returns a `FunctionCallOutput` to the orchestrator Conversation.

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 📦 Import Required Libraries and Setup Environment

Use the current Azure AI Projects models for versioned prompt agents and function tools, plus OpenAI response input types for returning tool outputs.

In [ ]:
import json
import os
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FunctionTool, PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)

env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find .env. Complete Lab 00 and place it in the repository root."
    )

load_dotenv(env_path)
tenant_id = os.environ.get("TENANT_ID")
ai_foundry_project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.environ.get("MODEL_DEPLOYMENT_NAME")
missing_variables = [
    name
    for name, value in {
        "TENANT_ID": tenant_id,
        "AI_FOUNDRY_PROJECT_ENDPOINT": ai_foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment,
    }.items()
    if not value
]
if missing_variables:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_variables)}")

print(f"📁 Environment loaded from: {env_path}")
print(f"🔑 Using Tenant ID: {tenant_id}")
print(f"🔗 Project Endpoint: {ai_foundry_project_endpoint}")

## 🎯 Define Specialist Agent Instructions

Now we'll define the instructions for each of our three specialist agents. Each agent has a specific role in the ticket triage process.

### Priority Assessment Agent

This agent analyzes tickets to determine their urgency level. It categorizes tickets as High, Medium, or Low priority based on their impact on users and business operations.

In [ ]:
# Priority agent definition
priority_agent_name = "priority_agent"
priority_agent_instructions = """
Assess how urgent a ticket is based on its description.

Respond with one of the following levels:
- High: User-facing or blocking issues
- Medium: Time-sensitive but not breaking anything
- Low: Cosmetic or non-urgent tasks

Only output the urgency level and a very brief explanation.
"""

### Team Assignment Agent

This agent determines which team should handle each ticket based on the technical domain and expertise required. It assigns tickets to Frontend, Backend, Infrastructure, or Marketing teams.

In [ ]:
# Team agent definition
team_agent_name = "team_agent"
team_agent_instructions = """
Decide which team should own each ticket.

Choose from the following teams:
- Frontend
- Backend
- Infrastructure
- Marketing

Base your answer on the content of the ticket. Respond with the team name and a very brief explanation.
"""

### Effort Estimation Agent

This agent estimates the amount of work required to resolve each ticket. It categorizes the effort as Small (1 day), Medium (2-3 days), or Large (multi-day/cross-team effort).

In [ ]:
# Effort agent definition
effort_agent_name = "effort_agent"
effort_agent_instructions = """
Estimate how much work each ticket will require.

Use the following scale:
- Small: Can be completed in a day
- Medium: 2-3 days of work
- Large: Multi-day or cross-team effort

Base your estimate on the complexity implied by the ticket. Respond with the effort level and a brief justification.
"""

### Orchestrator Agent Instructions

The main orchestrator agent coordinates all three specialist agents. It receives the ticket and uses the specialist agents as tools to provide a comprehensive triage analysis.

In [ ]:
triage_agent_instructions = """
Triage the user's support ticket by calling all three available functions:
assess_priority, assign_team, and estimate_effort. Use the same ticket text for
each function. After receiving all three tool outputs, provide a concise final
summary with Priority, Team, and Effort headings.
"""

## 🔗 Connect to Microsoft Foundry Agent Service

Create the project client and derive the OpenAI client used for Responses and Conversations.

In [ ]:
# Uses the `az login --use-device-code` session created in the authentication step.
credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(
    endpoint=ai_foundry_project_endpoint,
    credential=credential,
)
openai_client = project_client.get_openai_client()
print("✅ AIProjectClient and OpenAI client initialized")

## 🤖 Create the Multi-Agent System

Create the three specialist versions first. Then create an orchestrator version with one client-side `FunctionTool` for each specialist capability.

In [ ]:
def create_specialist(agent_name, instructions):
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_deployment,
            instructions=instructions,
        ),
    )
    print(f"✅ Created {agent.name}, version {agent.version}")
    return agent


priority_agent = create_specialist(priority_agent_name, priority_agent_instructions)
team_agent = create_specialist(team_agent_name, team_agent_instructions)
effort_agent = create_specialist(effort_agent_name, effort_agent_instructions)

ticket_parameters = {
    "type": "object",
    "properties": {
        "ticket": {
            "type": "string",
            "description": "The complete support ticket text to analyze.",
        }
    },
    "required": ["ticket"],
    "additionalProperties": False,
}

orchestrator_tools = [
    FunctionTool(
        name="assess_priority",
        description="Ask the priority specialist to assess ticket urgency.",
        parameters=ticket_parameters,
        strict=True,
    ),
    FunctionTool(
        name="assign_team",
        description="Ask the team specialist to select the owning team.",
        parameters=ticket_parameters,
        strict=True,
    ),
    FunctionTool(
        name="estimate_effort",
        description="Ask the effort specialist to estimate implementation effort.",
        parameters=ticket_parameters,
        strict=True,
    ),
]

triage_agent = project_client.agents.create_version(
    agent_name="triage-agent",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=triage_agent_instructions,
        tools=orchestrator_tools,
    ),
)
print(f"✅ Created orchestrator {triage_agent.name}, version {triage_agent.version}")

specialists = {
    "assess_priority": priority_agent,
    "assign_team": team_agent,
    "estimate_effort": effort_agent,
}

## 💬 Create the Orchestrator Conversation

The Conversation stores the orchestrator request, its function calls, each submitted function output, and the final triage answer.

In [ ]:
print("Creating orchestrator Conversation.")
triage_conversation = openai_client.conversations.create()
print(f"✅ Conversation ID: {triage_conversation.id}")

## 🎯 Execute Multi-Agent Ticket Triage

Now we'll run our multi-agent ticket triage system! This cell will:

1. **Send a ticket**: Submit a sample support ticket to the orchestrator agent
2. **Agent coordination**: The orchestrator will call each specialist agent for their assessment
3. **Display results**: Show the complete conversation flow and final triage analysis

Watch as the orchestrator agent intelligently uses the three specialist agents to provide a comprehensive ticket analysis including priority, team assignment, and effort estimation.

In [ ]:
def agent_reference(agent):
    return {
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    }


def invoke_specialist(agent, ticket):
    response = openai_client.responses.create(
        input=ticket,
        extra_body=agent_reference(agent),
    )
    return response.output_text


ticket = "Users can't reset their password from the mobile app."
print(f"📝 Ticket: {ticket}")

response = openai_client.responses.create(
    conversation=triage_conversation.id,
    input=ticket,
    extra_body=agent_reference(triage_agent),
)

for _ in range(10):
    function_outputs: ResponseInputParam = []

    for item in response.output:
        if item.type != "function_call":
            continue

        arguments = json.loads(item.arguments)
        specialist = specialists[item.name]
        specialist_output = invoke_specialist(specialist, arguments["ticket"])
        print(f"\n🔧 {item.name} -> {specialist_output}")
        function_outputs.append(
            FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=specialist_output,
            )
        )

    if not function_outputs:
        break

    response = openai_client.responses.create(
        conversation=triage_conversation.id,
        input=function_outputs,
        extra_body=agent_reference(triage_agent),
    )
else:
    raise RuntimeError("Orchestrator exceeded the maximum function-call rounds.")

print("\n📊 Final Multi-Agent Triage")
print("=" * 50)
print(response.output_text)

## 🧹 Clean Up Resources

Delete each prompt-agent version so tutorial resources do not accumulate in the project.

In [ ]:
openai_client.conversations.delete(conversation_id=triage_conversation.id)
print("🗑️ Deleted orchestrator Conversation")

for created_agent in [
    triage_agent,
    priority_agent,
    team_agent,
    effort_agent,
]:
    project_client.agents.delete_version(
        agent_name=created_agent.name,
        agent_version=created_agent.version,
    )
    print(f"🗑️ Deleted {created_agent.name}, version {created_agent.version}")

openai_client.close()
project_client.close()
credential.close()
print("✅ Cleanup completed!")